In [7]:
import pandas as pd

def replace_processing_status(input_csv, output_csv):
    """
    처리상태 열의 값을 조건에 따라 "반영" 또는 "미반영"으로 교체하고 저장.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        output_csv (str): 결과를 저장할 출력 CSV 파일 경로.
    """
    # CSV 파일 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    
    # 처리상태 열이 존재하는지 확인
    if "처리상태" in df.columns:
        # 반영으로 교체할 값들
        reflect_values = ["원안가결", "수정가결", "대안반영폐기", "수정안반영폐기"]
        # 미반영으로 교체할 값들
        not_reflect_values = ["임기만료폐기", "철회", "폐기"]
        
        # 처리상태 열 값 교체
        df["처리상태"] = df["처리상태"].apply(
            lambda x: "반영" if x in reflect_values else ("미반영" if x in not_reflect_values else x)
        )
        
        # 결과 저장
        df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"결과를 저장했습니다: {output_csv}")
    else:
        print("처리상태 열이 존재하지 않습니다.")

# 실행 예제
input_csv_path = "/Users/hyeongmin_k/law/로우법안데이터/data22_241210160645.csv"  # 입력 파일 경로
output_csv_path = "/Users/hyeongmin_k/law/로우법안데이터/data22_반영여부.csv"  # 결과를 저장할 파일 경로

replace_processing_status(input_csv_path, output_csv_path)


결과를 저장했습니다: /Users/hyeongmin_k/law/로우법안데이터/data22_반영여부.csv


In [1]:
import pandas as pd

# CSV 파일을 읽어들입니다.
df = pd.read_csv('의원데이터/assembly_members_20241127191947.csv', encoding='utf-8')  
# 조건에 맞는 행들을 저장할 리스트를 초기화합니다.
problematic_rows = []

# DataFrame의 각 행을 순회합니다.
for index, row in df.iterrows():
    activities = str(row["역대활동"]).replace(" ", "").split(",")  # 공백 제거 후 쉼표로 분리
    districts = str(row["선거구구분"]).split("/")  # 슬래시로 분리
    regions = str(row["선거구"]).split("/")  # 슬래시로 분리
    parties = str(row["정당"]).split("/")  # 슬래시로 분리

    # 리스트의 길이가 일치하지 않는 경우
    if len(activities) != len(districts) or len(activities) != len(regions) or len(activities) != len(parties):
        # 해당 행을 problematic_rows 리스트에 추가합니다.
        problematic_rows.append(row)

# 조건에 맞는 행들로 새로운 DataFrame을 생성합니다.
problematic_df = pd.DataFrame(problematic_rows)

# 새로운 CSV 파일로 저장합니다.
problematic_df.to_csv('의원데이터/문제맴버.csv', index=False, encoding='utf-8')


In [1]:
import pandas as pd

def filter_rows_by_term(input_csv, target_term, filtered_csv):
    """
    특정 제n대를 포함하는 행만 필터링하여 저장.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        target_term (str): 필터링할 대수 (예: "제21대").
        filtered_csv (str): 필터링된 결과를 저장할 CSV 파일 경로.
    """
    # CSV 파일 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    # 특정 제n대를 포함하는 행 필터링
    filtered_df = df[df["역대활동"].str.contains(target_term, na=False)]

    # 결과 저장
    filtered_df.to_csv(filtered_csv, index=False, encoding="utf-8-sig")
    print(f"'{target_term}'을 포함한 행만 저장했습니다: {filtered_csv}")


def update_term_based_on_reference(input_csv, reference_term, output_csv):
    """
    기준 대수를 설정하고, 그 이후의 대수가 몇 번 나오는지 확인하여 재선여부를 업데이트.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        reference_term (str): 기준 대수 (예: "제21대").
        output_csv (str): 결과를 저장할 출력 CSV 파일 경로.
    """
    # CSV 파일 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    # 재선여부 변경 로직
    def adjust_term(term, count_after_reference):
        term_mapping = {
            "초선": "초선",  # 초선은 그대로 유지
            "재선": "초선",  # 재선 -> 초선
            "3선": "재선",  # 3선 -> 재선
            "4선": "3선",  # 4선 -> 3선
            "5선": "4선",  # 5선 -> 4선
            "6선": "5선",  # 6선 -> 5선
            "7선": "6선",  # 7선 -> 6선
            "8선": "7선"   # 8선 -> 7선
        }

        # count_after_reference만큼 재선 단계를 줄이기
        for _ in range(count_after_reference):
            term = term_mapping.get(term, term)  # 단계 줄이기
        return term

    # 각 의원의 역대활동 확인 및 재선여부 수정
    def process_row(row):
        activities = str(row["역대활동"]).replace(" ", "").split(",")  # 쉼표로 분리하고 공백 제거
        # 기준 대수 이후 등장하는 대수 개수 계산
        count_after_reference = sum(1 for activity in activities if activity > reference_term)
        return adjust_term(row["재선여부"], count_after_reference)

    df["재선여부"] = df.apply(process_row, axis=1)

    # 결과를 새로운 CSV 파일로 저장
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"재선 여부가 업데이트된 CSV 파일이 저장되었습니다: {output_csv}")


# 실행 예제
input_csv_path = "/Users/hyeongmin_k/law/의원데이터/assembly_members_20241127191947.csv"  # 원본 CSV 파일 경로
target_term = "제22대"  # 필터링할 대수
filtered_csv_path = f"/Users/hyeongmin_k/law/의원데이터/{target_term}의원.csv"  # 필터링된 결과 저장 경로
updated_csv_path = f"/Users/hyeongmin_k/law/의원데이터/{target_term}의원.csv"  # 재선 여부 업데이트된 결과 저장 경로

# 1. 특정 제n대 포함 행만 필터링 후 저장
filter_rows_by_term(input_csv_path, target_term, filtered_csv_path)

# 2. 필터링된 데이터를 사용하여 재선 여부 업데이트
update_term_based_on_reference(filtered_csv_path, target_term, updated_csv_path)


'제22대'을 포함한 행만 저장했습니다: /Users/hyeongmin_k/law/의원데이터/제22대의원.csv
재선 여부가 업데이트된 CSV 파일이 저장되었습니다: /Users/hyeongmin_k/law/의원데이터/제22대의원.csv


In [2]:
import pandas as pd

def filter_and_reset_party_by_term(input_csv, target_term, output_csv):
    """
    특정 대수의 위치를 기반으로 선거구구분과 선거구는 새로운 열로 추가하고, 정당은 기존 열을 업데이트하는 코드.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        target_term (str): 찾고자 하는 대수 (예: "제21대").
        output_csv (str): 결과를 저장할 출력 CSV 파일 경로.
    """
    # 데이터 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    
    # 결과 저장을 위한 리스트
    filtered_districts = []
    filtered_regions = []

    for index, row in df.iterrows():
        # 역대활동, 선거구구분, 선거구, 정당 열을 분리
        activities = str(row["역대활동"]).replace(" ", "").split(",")  # 쉼표로 분리하고 공백 제거
        districts = str(row["선거구구분"]).split("/")  # 슬래시로 분리
        regions = str(row["선거구"]).split("/")  # 슬래시로 분리
        parties = str(row["정당"]).split("/")  # 슬래시로 분리

        if len(activities) != len(districts) or len(activities) != len(regions) or len(activities) != len(parties):
            # 길이가 다른 경우 빈 문자열로 처리
            filtered_districts.append("")
            filtered_regions.append("")
            df.at[index, "정당"] = ""  # 기존 정당 열 업데이트
            continue

        try:
            # target_term의 위치 찾기
            position = activities.index(target_term)
            # 해당 위치에 대응하는 데이터 가져오기
            district = districts[position]
            region = regions[position]
            party = parties[position]
        except ValueError:
            # target_term이 없으면 빈 문자열 반환
            district = ""
            region = ""
            party = ""

        filtered_districts.append(district)
        filtered_regions.append(region)
        df.at[index, "정당"] = party  # 기존 정당 열 업데이트

    # 결과를 새로운 열로 추가
    df["필터링된_선거구구분"] = filtered_districts
    df["필터링된_선거구"] = filtered_regions

    # 결과 저장
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"결과를 저장했습니다: {output_csv}")


# 실행 예제
target_term = "제22대"  # 찾고자 하는 대수
input_csv_path = f"/Users/hyeongmin_k/law/의원데이터/{target_term}의원.csv"  # 입력 파일 경로
output_csv_path =f"/Users/hyeongmin_k/law/의원데이터/{target_term}의원.csv"  # 결과를 저장할 파일 경로

filter_and_reset_party_by_term(input_csv_path, target_term, output_csv_path)


결과를 저장했습니다: /Users/hyeongmin_k/law/의원데이터/제22대의원.csv


In [4]:
import pandas as pd

def print_unique_parties(csv_file):
    """
    CSV 파일에서 정당 열의 고유 값(유니크 벨류)을 출력하는 함수

    Parameters:
        csv_file (str): 입력 CSV 파일 경로
    """
    try:
        # CSV 파일 읽기
        df = pd.read_csv(csv_file, encoding="utf-8-sig")

        # 정당 열의 유니크 값 추출
        if "정당" in df.columns:
            unique_parties = df["정당"].dropna().unique()  # NaN 값 제외
            print("정당의 유니크 벨류:")
            for party in unique_parties:
                print(party)
        else:
            print("정당 열이 CSV 파일에 없습니다.")

    except Exception as e:
        print(f"오류 발생: {e}")

# 실행 예제
target_term = "제22대"  # 찾고자 하는 대수

# 실행 예제
csv_file_path = f"/Users/hyeongmin_k/law/의원데이터/{target_term}_결과.csv"  # CSV 파일 경로
print_unique_parties(csv_file_path)


정당의 유니크 벨류:
조국혁신당
국민의힘
더불어민주당
국민의미래
더불어민주연합
새로운미래
진보당
개혁신당


In [32]:
import pandas as pd

def update_party_names(input_csv, output_csv):
    """
    CSV 파일의 정당 열에서 특정 정당 이름을 변경한 뒤 저장합니다.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        output_csv (str): 출력 CSV 파일 경로.
    """
    # CSV 파일 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    # 정당 이름 변경 규칙
    party_mapping = {
        "더불어민주연합": "더불어민주당",
        "국민의미래": "국민의힘",
    }

    # 정당열 이름이 "정당"이라고 가정
    if "정당" in df.columns:
        # 정당열 값을 매핑하여 변경
        df["정당"] = df["정당"].replace(party_mapping)

    # 수정된 데이터프레임 저장
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"정당 이름 수정 완료: {output_csv}")



# 실행 예제
target_term = "제22대"  # 찾고자 하는 대수
input_csv_path = f"/Users/hyeongmin_k/law/의원데이터/{target_term}_결과.csv"  # 입력 파일 경로
output_csv_path =f"/Users/hyeongmin_k/law/의원데이터/{target_term}_결과.csv"  # 결과를 저장할 파일 경로

update_party_names(input_csv_path, output_csv_path)


정당 이름 수정 완료: /Users/hyeongmin_k/law/의원데이터/제22대_결과.csv


In [1]:
import pandas as pd

def process_and_join(a_file, b_file, output_file):
    """
    의안별 CSV 파일(a)와 국회의원 리스트(b)를 조인하여 결과를 생성.
    대표발의자와 공동발의자를 ','로 분리하여 각각의 행으로 확장하고 기존 열을 유지.
    최종적으로 지정된 열만 남기고 저장.

    Parameters:
        a_file (str): 의안별 CSV 파일 경로.
        b_file (str): 국회의원 리스트 CSV 파일 경로.
        output_file (str): 결과 저장 파일 경로.
    """
    # 의안별 CSV 파일 (a) 읽기
    a_df = pd.read_csv(a_file, encoding="utf-8-sig")
    
    # 국회의원 리스트 CSV 파일 (b) 읽기
    b_df = pd.read_csv(b_file, encoding="utf-8-sig")

    # 데이터 타입 통일 (문자열로 변환)
    a_df["대표발의자"] = a_df["대표발의자"].astype(str)
    a_df["공동발의자"] = a_df["공동발의자"].astype(str)
    b_df["이름"] = b_df["이름"].astype(str)

    # 대표발의자와 공동발의자를 ','로 분리하여 행 확장
    expanded_rows = []
    for _, row in a_df.iterrows():
        representatives = str(row["대표발의자"]).split(",")
        co_sponsors = str(row["공동발의자"]).split(",")
        
        for representative in representatives:
            for co_sponsor in co_sponsors:
                new_row = row.to_dict()  # 기존 행을 복사
                new_row["대표발의자"] = representative.strip()  # 대표발의자를 개별화
                new_row["공동발의자"] = co_sponsor.strip()  # 공동발의자를 개별화
                expanded_rows.append(new_row)
    
    # 변환된 DataFrame 생성
    expanded_df = pd.DataFrame(expanded_rows)

    # b 파일에서 필요한 열만 선택
    b_columns = ["정당", "필터링된_선거구", "재선여부", "성별", "필터링된_선거구구분"]

    # 대표발의자에 대해 조인
    expanded_df = expanded_df.merge(
        b_df[["이름"] + b_columns].rename(columns=lambda x: f"대표발의자_{x}" if x != "이름" else "대표발의자"),
        left_on="대표발의자", 
        right_on="대표발의자",
        how="left"
    )

    # 공동발의자에 대해 조인
    expanded_df = expanded_df.merge(
        b_df[["이름"] + b_columns].rename(columns=lambda x: f"공동발의자_{x}" if x != "이름" else "공동발의자"),
        left_on="공동발의자", 
        right_on="공동발의자",
        how="left"
    )

    # 남길 열 지정
    columns_to_keep = [
        "의안번호", "소관위원회", "처리상태", "제안일",
        "대표발의자", "공동발의자",
        "대표발의자_정당", "대표발의자_필터링된_선거구", "대표발의자_재선여부",
        "대표발의자_성별", "대표발의자_필터링된_선거구구분",
        "공동발의자_정당", "공동발의자_필터링된_선거구", "공동발의자_재선여부",
        "공동발의자_성별", "공동발의자_필터링된_선거구구분"
    ]
    
    # 지정된 열만 선택
    final_df = expanded_df[columns_to_keep]



    print(final_df.head())


    # 결과를 CSV 파일로 저장
    final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"처리 완료! 결과가 {output_file}에 저장되었습니다.")


# 실행 예제
a_file = "/Users/hyeongmin_k/law/로우법안데이터/data22_반영여부.csv"  # 의안별 CSV 파일 경로
b_file = "/Users/hyeongmin_k/law/의원데이터/제22대_결과.csv"  # 국회의원 리스트 CSV 파일 경로
output_file = "/Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터.csv"  # 결과 저장 파일 경로

process_and_join(a_file, b_file, output_file)


      의안번호 소관위원회 처리상태         제안일 대표발의자 공동발의자 대표발의자_정당 대표발의자_필터링된_선거구  \
0  2206310   NaN  NaN  2024-12-09   이광희   박수현   더불어민주당      충북 청주시서원구   
1  2206310   NaN  NaN  2024-12-09   이광희   허성무   더불어민주당      충북 청주시서원구   
2  2206310   NaN  NaN  2024-12-09   이광희   박해철   더불어민주당      충북 청주시서원구   
3  2206310   NaN  NaN  2024-12-09   이광희   문금주   더불어민주당      충북 청주시서원구   
4  2206310   NaN  NaN  2024-12-09   이광희   이기헌   더불어민주당      충북 청주시서원구   

  대표발의자_재선여부 대표발의자_성별 대표발의자_필터링된_선거구구분 공동발의자_정당   공동발의자_필터링된_선거구 공동발의자_재선여부  \
0         초선        남              지역구   더불어민주당     충남 공주시부여군청양군         재선   
1         초선        남              지역구   더불어민주당        경남 창원시성산구         초선   
2         초선        남              지역구   더불어민주당          경기 안산시병         초선   
3         초선        남              지역구   더불어민주당  전남 고흥군보성군장흥군강진군         초선   
4         초선        남              지역구   더불어민주당          경기 고양시병         초선   

  공동발의자_성별 공동발의자_필터링된_선거구구분  
0        남              지역구  
1        남              지역

In [38]:
import pandas as pd

def filter_by_date(input_csv, output_csv, cutoff_date):
    """
    CSV 파일에서 특정 날짜 미만인 행만 필터링하여 저장.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        output_csv (str): 결과를 저장할 출력 CSV 파일 경로.
        cutoff_date (str): 기준 날짜 (예: "2022-04-18").
    """
    # CSV 파일 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    
    # 날짜 형식으로 변환
    df['제안일'] = pd.to_datetime(df['제안일'], errors='coerce')
    
    # 기준 날짜 미만 필터링
    cutoff_date = pd.to_datetime(cutoff_date)
    filtered_df = df[df['제안일'] >= cutoff_date]
    
    # 결과 저장
    filtered_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"필터링된 데이터를 저장했습니다: {output_csv}")

# 실행 예제
input_csv_path = "/Users/hyeongmin_k/law/의원데이터/제20대_법안별_전체데이터.csv"  # 입력 CSV 파일 경로
output_csv_path = "/Users/hyeongmin_k/law/의원데이터/제20대_법안별_전체데이터__2018-02-13_이후.csv"  # 결과를 저장할 파일 경로
cutoff_date = "2018-02-13"  # 기준 날짜

filter_by_date(input_csv_path, output_csv_path, cutoff_date)


필터링된 데이터를 저장했습니다: /Users/hyeongmin_k/law/의원데이터/제20대_법안별_전체데이터__2018-02-13_이후.csv


In [34]:
import pandas as pd

def create_gephi_edges_with_all_attributes(input_csv, output_node_csv, output_edge_csv):
    """
    Gephi에서 사용할 수 있는 노드 및 엣지 데이터를 생성.
    추가적으로 재선여부, 필터링된_선거구구분, 성별 속성을 포함.

    Parameters:
        input_csv (str): 입력 CSV 파일 경로.
        output_node_csv (str): Gephi 노드 데이터를 저장할 CSV 파일 경로.
        output_edge_csv (str): Gephi 엣지 데이터를 저장할 CSV 파일 경로.
    """
    # 데이터 읽기
    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    # Edge 데이터 생성
    edges = []
    for _, row in df.iterrows():
        representative = row["대표발의자"]
        co_sponsors = str(row["공동발의자"]).split(",")

        for co_sponsor in co_sponsors:
            co_sponsor = co_sponsor.strip()
            if co_sponsor:
                edges.append({
                    "Source": co_sponsor,
                    "Target": representative,
                    "Source_Party": row["공동발의자_정당"],
                    "Target_Party": row["대표발의자_정당"],
                    "Source_Reselection": row["공동발의자_재선여부"],
                    "Target_Reselection": row["대표발의자_재선여부"],
                    "Source_District_Type": row["공동발의자_필터링된_선거구구분"],
                    "Target_District_Type": row["대표발의자_필터링된_선거구구분"],
                    "Source_Gender": row["공동발의자_성별"],
                    "Target_Gender": row["대표발의자_성별"],
                    "Source_Role": "Co-Sponsor",
                    "Target_Role": "Representative"
                })

    # DataFrame으로 변환
    edges_df = pd.DataFrame(edges)

    # 노드 데이터 생성 (Source와 Target 노드 합집합으로 생성)
    nodes = set(edges_df["Source"]).union(set(edges_df["Target"]))
    node_data = []
    for node in nodes:
        # Source 속성
        source_row = df[df["공동발의자"] == node].iloc[0] if node in df["공동발의자"].values else None
        # Target 속성
        target_row = df[df["대표발의자"] == node].iloc[0] if node in df["대표발의자"].values else None

        node_data.append({
            "Id": node,
            "Label": node,
            "Party": target_row["대표발의자_정당"] if target_row is not None else (
                source_row["공동발의자_정당"] if source_row is not None else None),
            "Reselection": target_row["대표발의자_재선여부"] if target_row is not None else (
                source_row["공동발의자_재선여부"] if source_row is not None else None),
            "District_Type": target_row["대표발의자_필터링된_선거구구분"] if target_row is not None else (
                source_row["공동발의자_필터링된_선거구구분"] if source_row is not None else None),
            "Gender": target_row["대표발의자_성별"] if target_row is not None else (
                source_row["공동발의자_성별"] if source_row is not None else None),
            "Role": "Representative" if target_row is not None else "Co-Sponsor"
        })

    # DataFrame으로 변환
    nodes_df = pd.DataFrame(node_data)

    # Gephi에 맞는 형식으로 저장
    nodes_df.to_csv(output_node_csv, index=False, encoding="utf-8-sig")
    edges_df.to_csv(output_edge_csv, index=False, encoding="utf-8-sig")

    print(f"Gephi 노드 데이터를 저장했습니다: {output_node_csv}")
    print(f"Gephi 엣지 데이터를 저장했습니다: {output_edge_csv}")


# 실행 예제
input_csv_path = "/Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터.csv"  # 입력 CSV 파일 경로
output_node_csv_path = "/Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터_node.csv"  # Gephi 노드 파일 경로
output_edge_csv_path = "/Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터_edge.csv"  # Gephi 엣지 파일 경로

create_gephi_edges_with_all_attributes(input_csv_path, output_node_csv_path, output_edge_csv_path)


Gephi 노드 데이터를 저장했습니다: /Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터_node.csv
Gephi 엣지 데이터를 저장했습니다: /Users/hyeongmin_k/law/의원데이터/제22대_법안별_전체데이터_edge.csv


In [8]:
import pandas as pd

def update_party_information(a_csv, b_csv, output_csv):
    """
    a.csv의 대표발의자 및 공동발의자 열과 b.csv의 이름 열을 비교하여
    a.csv의 공동발의자_정당 및 대표발의자_정당 값을 b.csv의 정당으로 업데이트.

    Parameters:
        a_csv (str): a.csv 파일 경로.
        b_csv (str): b.csv 파일 경로.
        output_csv (str): 결과를 저장할 CSV 파일 경로.
    """
    # a와 b CSV 파일 읽기
    a_df = pd.read_csv(a_csv, encoding="utf-8-sig")
    b_df = pd.read_csv(b_csv, encoding="utf-8-sig")

    # b의 이름과 정당 정보를 딕셔너리로 생성
    name_to_party = dict(zip(b_df["이름"], b_df["정당"]))

    # 대표발의자_정당 업데이트
    a_df["대표발의자_정당"] = a_df["대표발의자"].map(name_to_party).fillna(a_df["대표발의자_정당"])

    # 공동발의자_정당 업데이트
    def update_co_sponsors_party(co_sponsors):
        co_sponsors = str(co_sponsors).split(",")  # 공동발의자들을 분리
        updated_parties = [name_to_party.get(name.strip(), "") for name in co_sponsors]
        return ",".join(updated_parties)

    a_df["공동발의자_정당"] = a_df["공동발의자"].apply(update_co_sponsors_party)

    # 결과 저장
    a_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"정당 정보가 업데이트된 파일이 저장되었습니다: {output_csv}")


# 실행 예제
a_csv_path = "제21대_국민의당_편입_법안별.csv"  # a.csv 파일 경로
b_csv_path = "제21대_보통의원.csv"  # b.csv 파일 경로
output_csv_path = "제21대_전체데이터.csv"  # 결과를 저장할 파일 경로

update_party_information(a_csv_path, b_csv_path, output_csv_path)


정당 정보가 업데이트된 파일이 저장되었습니다: 제21대_전체데이터.csv
